# 09 多频率择时残差网络 (MFTR) Alpha因子挖掘

## 论文参考

- **作者**: Yihuang Guo (郭一皇)
- **年份**: 2023
- **标题**: *Mining Alpha Factors using Multi-Frequency Timing Residual Networks*
- **关键成果**: 年化收益率 26.92%

### 摘要

不同投资者在不同时间尺度上做出决策：短线交易者关注日级别数据，中线关注周级别，
长线关注月级别。本文提出多频率择时残差网络(MFTR)，同时从日频、周频(5日)、月频(20日)
三个时间尺度提取特征，通过残差连接融合多频率信息，挖掘具有预测能力的Alpha因子。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 多频率数据聚合

给定日频数据 $\{x_t\}$，构建三种频率的聚合：

- **日频 (1D)**: 原始日线数据 $x_t^{(1)} = x_t$
- **周频 (5D)**: 5日滑窗聚合 $x_t^{(5)} = \text{Agg}(x_{t-4}, ..., x_t)$
- **月频 (20D)**: 20日滑窗聚合 $x_t^{(20)} = \text{Agg}(x_{t-19}, ..., x_t)$

聚合操作 $\text{Agg}$ 包括均值、标准差、最大值、最小值等统计量。

### 多频率分支网络

每个频率分支 $k \in \{1, 5, 20\}$：

$$h^{(k)} = \text{LSTM}_k\left(\mathbf{x}^{(k)}_{t-T+1:t}\right)$$

### 残差连接

分支间通过残差连接融合信息：

$$\tilde{h}^{(k)} = h^{(k)} + \text{FC}_{\text{res}}\left(h^{(k')}\right)$$

其中低频信息通过残差连接补充到高频分支，捕获多尺度的市场结构。

### 融合与预测

$$\hat{y} = \sigma\left(\mathbf{W} \cdot [\tilde{h}^{(1)}; \tilde{h}^{(5)}; \tilde{h}^{(20)}] + b\right)$$

### 残差学习的优势

$$\mathcal{F}(x) = \mathcal{H}(x) - x$$

残差连接让网络学习增量信息 $\mathcal{F}(x)$ 而非直接学习映射 $\mathcal{H}(x)$，
在深度网络中有效缓解梯度消失问题。

In [ ]:
# ============================================================
# Cell 3: 动画 - 多频率分支 + 残差连接处理流程
# ============================================================
import numpy as np
import plotly.graph_objects as go

def animate_mftr_architecture():
    """动画展示MFTR多频率分支并行处理和残差连接"""
    frames = []
    step_titles = [
        '1. 原始日频数据输入',
        '2. 多频率聚合 (1D / 5D / 20D)',
        '3. 各分支LSTM独立编码',
        '4. 残差连接: 低频信息注入高频',
        '5. 特征融合与预测输出',
    ]

    # 分支颜色
    colors = {'1D': '#F44336', '5D': '#FF9800', '20D': '#2196F3'}
    branch_y = {'1D': 3, '5D': 1.5, '20D': 0}

    for step in range(5):
        data = []

        # 输入框
        if step >= 0:
            data.append(go.Scatter(
                x=[0.5], y=[1.5], mode='markers+text',
                marker=dict(size=40, color='#9E9E9E', symbol='square'),
                text=['Daily<br>Data'], textposition='middle center',
                textfont=dict(size=9, color='white'),
                showlegend=False, hoverinfo='none'
            ))

        # 频率聚合块
        if step >= 1:
            for label, y in branch_y.items():
                alpha = 0.8 if step >= 1 else 0.3
                data.append(go.Scatter(
                    x=[2.5], y=[y], mode='markers+text',
                    marker=dict(size=35, color=colors[label], symbol='square', opacity=alpha),
                    text=[f'Agg<br>{label}'], textposition='middle center',
                    textfont=dict(size=9, color='white'),
                    showlegend=False, hoverinfo='none'
                ))
                # 输入 -> 聚合
                data.append(go.Scatter(
                    x=[0.9, 2.1], y=[1.5, y], mode='lines',
                    line=dict(color='gray', width=1.5), showlegend=False, hoverinfo='none'
                ))

        # LSTM分支
        if step >= 2:
            for label, y in branch_y.items():
                glow = 1.0 if step == 2 else 0.8
                data.append(go.Scatter(
                    x=[4.5], y=[y], mode='markers+text',
                    marker=dict(size=38, color=colors[label], symbol='square', opacity=glow),
                    text=[f'LSTM<br>{label}'], textposition='middle center',
                    textfont=dict(size=9, color='white'),
                    showlegend=False, hoverinfo='none'
                ))
                data.append(go.Scatter(
                    x=[2.9, 4.1], y=[y, y], mode='lines',
                    line=dict(color=colors[label], width=2), showlegend=False, hoverinfo='none'
                ))

        # 残差连接
        if step >= 3:
            # 20D -> 5D residual
            data.append(go.Scatter(
                x=[4.5, 5.5, 5.5, 4.5], y=[0, 0, 1.5, 1.5],
                mode='lines', line=dict(color='#4CAF50', width=2, dash='dot'),
                showlegend=False, hoverinfo='none'
            ))
            data.append(go.Scatter(
                x=[5.5], y=[0.75], mode='text',
                text=['Res+'], textfont=dict(size=10, color='#4CAF50'),
                showlegend=False, hoverinfo='none'
            ))
            # 5D -> 1D residual
            data.append(go.Scatter(
                x=[4.5, 5.8, 5.8, 4.5], y=[1.5, 1.5, 3, 3],
                mode='lines', line=dict(color='#4CAF50', width=2, dash='dot'),
                showlegend=False, hoverinfo='none'
            ))
            data.append(go.Scatter(
                x=[5.8], y=[2.25], mode='text',
                text=['Res+'], textfont=dict(size=10, color='#4CAF50'),
                showlegend=False, hoverinfo='none'
            ))

        # 融合与输出
        if step >= 4:
            data.append(go.Scatter(
                x=[7], y=[1.5], mode='markers+text',
                marker=dict(size=45, color='#9C27B0', symbol='square'),
                text=['Concat<br>+FC'], textposition='middle center',
                textfont=dict(size=9, color='white'),
                showlegend=False, hoverinfo='none'
            ))
            for label, y in branch_y.items():
                data.append(go.Scatter(
                    x=[4.9, 6.6], y=[y, 1.5], mode='lines',
                    line=dict(color=colors[label], width=1.5), showlegend=False, hoverinfo='none'
                ))
            data.append(go.Scatter(
                x=[8.5], y=[1.5], mode='markers+text',
                marker=dict(size=30, color='#333', symbol='circle'),
                text=['\u03c3'], textposition='middle center',
                textfont=dict(size=14, color='white'),
                showlegend=False, hoverinfo='none'
            ))
            data.append(go.Scatter(
                x=[7.4, 8.2], y=[1.5, 1.5], mode='lines',
                line=dict(color='#333', width=2), showlegend=False, hoverinfo='none'
            ))

        frames.append(go.Frame(data=data, name=step_titles[step]))

    fig = go.Figure(data=frames[0].data, frames=frames)
    fig.update_layout(
        title=dict(text='MFTR 多频率择时残差网络架构动画'),
        xaxis=dict(visible=False, range=[-0.5, 9.5]),
        yaxis=dict(visible=False, range=[-1.5, 4.5]),
        height=450, width=950,
        plot_bgcolor='white',
        updatemenus=[dict(type='buttons', x=0.1, y=1.12, buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 1500}, 'transition': {'duration': 400}}]),
            dict(label='\u23f8 暂停', method='animate',
                 args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
            active=0, x=0.05, len=0.9,
        )],
    )
    return fig

fig = animate_mftr_architecture()
fig.show()

In [ ]:
# ============================================================
# Cell 4: 导入与环境设置
# ============================================================
import sys, os, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from shared.data_fetcher import get_stock_daily
from shared.backtest_engine import Backtester
from shared.visualization import plot_full_report, set_chinese_font
from shared.factors import momentum, volatility, rsi, macd, bollinger_bands, price_to_ma

set_chinese_font()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')

In [ ]:
# ============================================================
# Cell 5: 数据获取
# ============================================================
SYMBOL = '600519'  # 贵州茅台
df = get_stock_daily(SYMBOL, start_date='20200601', end_date='20241231')
print(f'数据形状: {df.shape}')
print(f'日期范围: {df.index[0]} ~ {df.index[-1]}')
df.tail(3)

In [ ]:
# ============================================================
# Cell 6: 特征工程 - 多频率聚合
# ============================================================
LOOKBACK = 30

def compute_base_features(df):
    """计算基础特征"""
    feat = pd.DataFrame(index=df.index)
    feat['ret'] = df['close'].pct_change()
    feat['vol'] = df['close'].pct_change().rolling(10).std()
    feat['rsi'] = rsi(df['close'], 14)
    macd_df = macd(df['close'])
    feat['macd_hist'] = macd_df['histogram']
    bb = bollinger_bands(df['close'])
    feat['bb_pctb'] = bb['bb_pctb']
    feat['log_volume'] = np.log1p(df['volume'])
    if 'turnover' in df.columns:
        feat['turnover'] = df['turnover']
    feat['hl_range'] = (df['high'] - df['low']) / df['close']
    return feat


def multi_freq_aggregate(base_feat, windows=[1, 5, 20]):
    """多频率聚合: 对每个基础特征计算不同窗口的统计量"""
    freq_data = {}
    for w in windows:
        agg = pd.DataFrame(index=base_feat.index)
        for col in base_feat.columns:
            if w == 1:
                agg[f'{col}'] = base_feat[col]
            else:
                agg[f'{col}_mean_{w}d'] = base_feat[col].rolling(w).mean()
                agg[f'{col}_std_{w}d'] = base_feat[col].rolling(w).std()
                agg[f'{col}_max_{w}d'] = base_feat[col].rolling(w).max()
                agg[f'{col}_min_{w}d'] = base_feat[col].rolling(w).min()
        freq_data[w] = agg
    return freq_data


def build_mftr_dataset(df, lookback=30):
    """构建MFTR多频率数据集"""
    base = compute_base_features(df)
    freq_data = multi_freq_aggregate(base, windows=[1, 5, 20])

    # 对齐：取所有频率都有数据的日期
    common_idx = base.index.copy()
    for w, fd in freq_data.items():
        fd.dropna(inplace=True)
        common_idx = common_idx.intersection(fd.index)

    # 标签
    target = (df['close'].shift(-1) > df['close']).astype(int)
    target = target.reindex(common_idx)

    # 去掉NaN
    valid_mask = ~target.isna()
    common_idx = common_idx[valid_mask]
    target = target.loc[common_idx].values.astype(np.float32)

    # 构建滑窗序列 - 每个频率一组
    X_freq = {w: [] for w in [1, 5, 20]}
    y_list = []
    dates_out = []

    freq_arrays = {w: freq_data[w].loc[common_idx].values for w in [1, 5, 20]}
    n = len(common_idx)

    for i in range(lookback, n):
        for w in [1, 5, 20]:
            X_freq[w].append(freq_arrays[w][i-lookback:i])
        y_list.append(target[i])
        dates_out.append(common_idx[i])

    result = {
        w: np.array(X_freq[w], dtype=np.float32) for w in [1, 5, 20]
    }
    return result, np.array(y_list, dtype=np.float32), dates_out


X_freq, y, dates = build_mftr_dataset(df, LOOKBACK)
for w in [1, 5, 20]:
    print(f'频率 {w}D: shape = {X_freq[w].shape}')
print(f'标签: {len(y)}, 涨={int(y.sum())}, 跌={int(len(y) - y.sum())}')

In [ ]:
# ============================================================
# Cell 7: PyTorch MFTR 模型与训练
# ============================================================

class FreqBranch(nn.Module):
    """单个频率分支: LSTM编码器"""
    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=1,
                            batch_first=True, dropout=0)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.norm(out[:, -1, :])


class ResidualBlock(nn.Module):
    """残差连接: 从源分支到目标分支"""
    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, target, source):
        residual = self.fc(source)
        return self.norm(target + residual)


class MFTR(nn.Module):
    """多频率择时残差网络"""
    def __init__(self, input_dims, hidden_dim=32):
        super().__init__()
        # 三个频率分支
        self.branch_1d = FreqBranch(input_dims[1], hidden_dim)
        self.branch_5d = FreqBranch(input_dims[5], hidden_dim)
        self.branch_20d = FreqBranch(input_dims[20], hidden_dim)

        # 残差连接: 低频 -> 高频
        self.res_20_to_5 = ResidualBlock(hidden_dim)
        self.res_5_to_1 = ResidualBlock(hidden_dim)

        # 融合预测头
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 3, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, x_1d, x_5d, x_20d):
        h_1d = self.branch_1d(x_1d)
        h_5d = self.branch_5d(x_5d)
        h_20d = self.branch_20d(x_20d)

        # 残差: 20D -> 5D -> 1D
        h_5d = self.res_20_to_5(h_5d, h_20d)
        h_1d = self.res_5_to_1(h_1d, h_5d)

        # 拼接融合
        merged = torch.cat([h_1d, h_5d, h_20d], dim=-1)
        return self.head(merged).squeeze(-1)


# --- 数据划分与标准化 ---
TRAIN_END = pd.Timestamp('2023-12-31')
dates_ts = pd.DatetimeIndex(dates)
train_mask = dates_ts <= TRAIN_END
test_mask = dates_ts > TRAIN_END

scalers = {}
X_train_freq, X_test_freq = {}, {}
input_dims = {}

for w in [1, 5, 20]:
    sc = StandardScaler()
    n, seq, feat = X_freq[w].shape
    input_dims[w] = feat
    train_flat = X_freq[w][train_mask].reshape(-1, feat)
    sc.fit(train_flat)
    scalers[w] = sc

    train_s = sc.transform(X_freq[w][train_mask].reshape(-1, feat)).reshape(-1, seq, feat)
    test_s = sc.transform(X_freq[w][test_mask].reshape(-1, feat)).reshape(-1, seq, feat)
    # 替换NaN/Inf
    train_s = np.nan_to_num(train_s, 0)
    test_s = np.nan_to_num(test_s, 0)
    X_train_freq[w] = torch.FloatTensor(train_s)
    X_test_freq[w] = torch.FloatTensor(test_s)

y_train = torch.FloatTensor(y[train_mask])
y_test = y[test_mask]
test_dates_out = dates_ts[test_mask]

print(f'训练: {len(y_train)} 样本, 测试: {len(y_test)} 样本')
print(f'各频率特征维度: {input_dims}')

train_dataset = TensorDataset(
    X_train_freq[1], X_train_freq[5], X_train_freq[20], y_train
)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# --- 训练 ---
model = MFTR(input_dims=input_dims, hidden_dim=32).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

EPOCHS = 50
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    n_s = 0
    for x1, x5, x20, lab in train_loader:
        x1, x5, x20, lab = x1.to(device), x5.to(device), x20.to(device), lab.to(device)
        optimizer.zero_grad()
        pred = model(x1, x5, x20)
        loss = criterion(pred, lab)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item() * len(lab)
        n_s += len(lab)
    scheduler.step()

    avg_loss = epoch_loss / max(n_s, 1)
    train_losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS}  Loss: {avg_loss:.4f}')

# --- 预测 ---
model.eval()
with torch.no_grad():
    probs = model(
        X_test_freq[1].to(device),
        X_test_freq[5].to(device),
        X_test_freq[20].to(device)
    ).cpu().numpy()

test_preds = (probs > 0.5).astype(int)
accuracy = (test_preds == y_test).mean()
print(f'\n测试集准确率: {accuracy:.4f}')

In [ ]:
# ============================================================
# Cell 8: 回测
# ============================================================
signals = pd.Series(test_preds, index=test_dates_out, name='signal')
test_prices = df.loc[test_dates_out, 'close']

bt = Backtester(initial_capital=1_000_000, t_plus_1=True)
result = bt.run(test_prices, signals)

print('回测绩效指标:')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 可视化
# ============================================================
import matplotlib.pyplot as plt

# 1. 训练损失
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, color='#2196F3', linewidth=1.5)
ax.set_title('MFTR 训练损失', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE Loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2. 多频率特征示例
base = compute_base_features(df)
sample = base['ret'].dropna()[-200:]
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
axes[0].plot(sample.index, sample.values, color='#F44336', linewidth=1)
axes[0].set_title('日频收益率 (1D)', fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].plot(sample.index, sample.rolling(5).mean().values, color='#FF9800', linewidth=1)
axes[1].set_title('5日均值收益率 (5D聚合)', fontsize=11)
axes[1].grid(True, alpha=0.3)

axes[2].plot(sample.index, sample.rolling(20).mean().values, color='#2196F3', linewidth=1)
axes[2].set_title('20日均值收益率 (20D聚合)', fontsize=11)
axes[2].grid(True, alpha=0.3)

fig.suptitle('多频率数据聚合示例', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# 3. 完整回测报告
plot_full_report(result, strategy_name='MFTR (Guo 2023)')

## 结果分析

### 模型架构

- **三频率分支**: 日频(1D)、周频(5D)、月频(20D)
- **聚合特征**: 均值、标准差、最大值、最小值统计量
- **残差连接**: 20D -> 5D -> 1D，低频趋势信息逐级注入高频
- **融合头**: 拼接三分支输出 -> FC(64) -> FC(32) -> Sigmoid

### 核心创新

1. **多频率视角**: 模拟不同时间尺度投资者的决策逻辑
2. **残差连接方向**: 低频(长期趋势) -> 高频(短期信号)，符合市场直觉
3. **Alpha因子挖掘**: 多频率融合提取的特征可作为新的Alpha因子

### 与论文对比

- 论文报告年化收益 26.92%，实际表现取决于市场环境和具体参数
- 本实现为简化版，原文可能包含更复杂的频率聚合和注意力机制